# **How to Build a Complete Futures Database in Python Using Norgate Data**

---

**Author:**  
[**Mohamed Gabriel**](https://x.com/M_S_Gabriel)  
*Software Engineer at Concretum Group*

---

This notebook demonstrates how to **download and structure all futures data from Norgate Data** using Python. The workflow produces four Parquet files covering contract specifications, individual contract price histories (including expired contracts), back-adjusted continuous futures with unadjusted prices merged in, and volume decomposition by contract expiry.

A similar procedure is used for constructing futures datasets in research by **Concretum Group**.

For detailed explanations, please refer to the full blog post or reach the authors via:

📧 **Email:** [info@concretumgroup.com](mailto:info@concretumgroup.com)  
🌐 **Website:** [www.concretumgroup.com](https://www.concretumgroup.com)

---

Step 0: Install Required Dependencies

In [ ]:
# %pip install norgatedata pandas pyarrow numpy

Step 1: Import Required Libraries and Set Up Logging

In [ ]:
import os
import re
import logging

import numpy as np
import norgatedata
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, Dict

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

print(f"norgatedata version: {norgatedata.__version__}")
print(f"pandas version: {pd.__version__}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────
# Adjust these to fit your needs

START_DATE = pd.Timestamp('1950-01-01')
END_DATE   = pd.Timestamp('2028-12-31')

MAX_WORKERS = 10                                        # threads for parallel downloads
PADDING     = norgatedata.PaddingType.ALLMARKETDAYS     # pad to all market trading days
TS_FORMAT   = 'pandas-dataframe'

# ── Output ─────────────────────────────────────────────────────
OUTPUT_DIR = 'norgate_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_CONTRACT_SPECS       = os.path.join(OUTPUT_DIR, '01_contract_specs.parquet')
PATH_INDIVIDUAL_OHLCV     = os.path.join(OUTPUT_DIR, '02_individual_contracts_ohlcv.parquet')
PATH_CONTINUOUS_ADJ_UNADJ   = os.path.join(OUTPUT_DIR, '03_continuous_adjusted_and_unadjusted.parquet')
PATH_CONTINUOUS_WITH_VOLUMES = os.path.join(OUTPUT_DIR, '04_continuous_with_volume_decomposition.parquet')

Step 2: Define Helper Function to Save Data

In [ ]:
def save_to_parquet(df: pd.DataFrame, filepath: str) -> None:
    df.to_parquet(filepath, engine='pyarrow', index=False)
    logging.info(f"Saved {len(df):,} rows to '{filepath}'.")

Step 3: Download Contract Specifications (Symbol Metadata)

In [ ]:
def get_symbol_metadata(symbol: str) -> Optional[Dict]:
    """Fetch contract specifications for a single continuous futures symbol."""
    data: Dict = {'Symbol': symbol}
    try:
        data['Name']     = norgatedata.security_name(symbol)
    except Exception:
        data['Name']     = None
    try:
        data['Exchange'] = norgatedata.exchange_name(symbol)
    except Exception:
        data['Exchange'] = None
    try:
        data['Group']    = norgatedata.classification_at_level(
            symbol,
            schemename='NorgateDataFuturesClassification',
            classificationresulttype='Name',
            level=1,
        )
    except Exception:
        data['Group']    = None
    try:
        data['Contract Size'] = norgatedata.point_value(symbol)
    except Exception:
        data['Contract Size'] = None
    try:
        data['Tick Size'] = norgatedata.tick_size(symbol)
    except Exception:
        data['Tick Size'] = None

    ts = data['Tick Size']
    cs = data['Contract Size']
    data['Tick Value']  = (ts * cs) if (ts is not None and cs is not None) else None
    data['Point Value'] = cs

    try:
        data['Currency'] = norgatedata.currency(symbol)
    except Exception:
        data['Currency'] = None
    try:
        data['Margin']   = norgatedata.margin(symbol)
    except Exception:
        data['Margin']   = None

    return data

In [ ]:
# Retrieve all continuous futures symbols
cont_symbols = norgatedata.database_symbols('Continuous Futures')
print(f"Continuous Futures symbols: {len(cont_symbols)}")

# Download metadata in parallel
metadata_rows = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futs = {pool.submit(get_symbol_metadata, s): s for s in cont_symbols}
    for f in as_completed(futs):
        result = f.result()
        if result:
            metadata_rows.append(result)

metadata_df = pd.DataFrame(metadata_rows)
save_to_parquet(metadata_df, PATH_CONTRACT_SPECS)
metadata_df.head(10)

Step 4: Download All Individual Futures Contracts (Including Expired)

In [ ]:
def download_individual_contract(symbol: str) -> Optional[pd.DataFrame]:
    """Download OHLCV data for one individual futures contract."""
    try:
        pricedata = norgatedata.price_timeseries(
            symbol,
            padding_setting=PADDING,
            start_date=START_DATE,
            end_date=END_DATE,
            timeseriesformat=TS_FORMAT,
        )

        if 'Date' not in pricedata.columns:
            pricedata.reset_index(inplace=True)

        pricedata['Symbol'] = symbol
        return pricedata

    except Exception as e:
        logging.error(f"Error downloading {symbol}: {e}")
        return None

In [ ]:
# Retrieve all individual futures symbols (including expired)
individual_symbols = norgatedata.database_symbols('Futures')
print(f"Individual Futures symbols (incl. expired): {len(individual_symbols)}")

# Download in parallel
all_individual = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futs = {pool.submit(download_individual_contract, s): s for s in individual_symbols}
    for f in as_completed(futs):
        result = f.result()
        if result is not None:
            all_individual.append(result)

individual_df = pd.concat(all_individual, ignore_index=True)
save_to_parquet(individual_df, PATH_INDIVIDUAL_OHLCV)
print(f"Shape: {individual_df.shape}")
individual_df.head(10)

Step 5: Download Back-Adjusted Continuous Futures (With Unadjusted Prices)

In [ ]:
def download_continuous_contract(symbol: str) -> Optional[pd.DataFrame]:
    """Download price data for one continuous futures symbol."""
    try:
        pricedata = norgatedata.price_timeseries(
            symbol,
            padding_setting=PADDING,
            start_date=START_DATE,
            end_date=END_DATE,
            timeseriesformat=TS_FORMAT,
        )
        if 'Date' not in pricedata.columns:
            pricedata.reset_index(inplace=True)
        pricedata['Symbol'] = symbol
        return pricedata
    except Exception as e:
        logging.error(f"Error downloading {symbol}: {e}")
        return None


def merge_adjusted_unadjusted(data: pd.DataFrame) -> pd.DataFrame:
    """Merge unadjusted close/volume into back-adjusted (_CCB) rows."""
    adjusted   = data[data['Symbol'].str.endswith('_CCB')].copy()
    unadjusted = data[~data['Symbol'].str.endswith('_CCB')].copy()

    adjusted['BaseSymbol']   = adjusted['Symbol'].str.replace('_CCB', '', regex=False)
    unadjusted['BaseSymbol'] = unadjusted['Symbol']

    merged = pd.merge(
        adjusted,
        unadjusted[['Date', 'Close', 'Volume', 'BaseSymbol']],
        on=['Date', 'BaseSymbol'],
        how='left',
        suffixes=('', ' Unadjusted'),
    )
    merged.drop(columns=['BaseSymbol'], inplace=True)
    return merged

In [ ]:
# Download all continuous futures in parallel
all_continuous = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futs = {pool.submit(download_continuous_contract, s): s for s in cont_symbols}
    for f in as_completed(futs):
        result = f.result()
        if result is not None:
            all_continuous.append(result)

continuous_df = pd.concat(all_continuous, ignore_index=True)
back_adjusted_df = merge_adjusted_unadjusted(continuous_df)
save_to_parquet(back_adjusted_df, PATH_CONTINUOUS_ADJ_UNADJ)
print(f"Shape: {back_adjusted_df.shape}")
back_adjusted_df.head(10)

Step 6: Volume Decomposition (First & Second Contract Volumes)

In [ ]:
MONTH_CODES = {
    'F': 1, 'G': 2, 'H': 3, 'J': 4, 'K': 5, 'M': 6,
    'N': 7, 'Q': 8, 'U': 9, 'V': 10, 'X': 11, 'Z': 12
}


def parse_individual_symbol(symbol: str):
    """Parse '6A-2025M' into ('6A', Timestamp('2025-06-01'))."""
    match = re.match(r'^(.+)-(\d{4})([FGHJKMNQUVXZ])$', symbol)
    if match:
        base = match.group(1)
        year = int(match.group(2))
        month = MONTH_CODES[match.group(3)]
        return base, pd.Timestamp(year=year, month=month, day=1)
    return None, None


def get_base_from_continuous(symbol: str, base_map: dict) -> str:
    """
    Extract base commodity code, checking against the base_map to resolve ambiguity.

    Tries the full code first (e.g. 'SO3' from '&SO3_CCB'), then falls back to
    stripping trailing digits (e.g. 'SO') for continuation symbols like '&6A2_CCB'.
    """
    raw = symbol.lstrip('&')
    raw = re.sub(r'_CC[A-Z]$', '', raw)

    if raw in base_map:
        return raw

    stripped = re.sub(r'\d+$', '', raw)
    if stripped in base_map:
        return stripped

    return raw


def build_base_map(symbols: list) -> dict:
    """Build {base_code: [(symbol, expiry), ...]} sorted by expiry date."""
    base_map = {}
    skipped = 0
    for sym in symbols:
        base, expiry = parse_individual_symbol(sym)
        if base:
            base_map.setdefault(base, []).append((sym, expiry))
        else:
            skipped += 1
    if skipped:
        logging.warning(f"Could not parse {skipped} individual futures symbols.")
    for base in base_map:
        base_map[base].sort(key=lambda x: x[1])
    logging.info(f"Built base map with {len(base_map)} commodities.")
    return base_map


def extract_contract_volumes(df: pd.DataFrame, symbols: set) -> dict:
    """Extract {symbol: DataFrame[Date, Volume]} from the already-downloaded individual_df."""
    filtered = df[df['Symbol'].isin(symbols)][['Date', 'Volume', 'Symbol']]
    return {
        sym: group[['Date', 'Volume']].reset_index(drop=True)
        for sym, group in filtered.groupby('Symbol')
    }


def compute_first_second_volume(dates, contracts, contract_volumes):
    """
    For each date, identify the first-to-expire and second-to-expire contracts
    and return their volumes.
    """
    available = [(sym, exp) for sym, exp in contracts if sym in contract_volumes]
    if not available:
        return None

    dates_df = pd.DataFrame({'Date': dates})
    for sym, _ in available:
        vol_df = contract_volumes[sym].rename(columns={'Volume': sym})
        dates_df = dates_df.merge(vol_df, on='Date', how='left')

    contract_cols = [sym for sym, _ in available]
    vol_array = dates_df[contract_cols].values.astype(float)
    num_dates, num_contracts = vol_array.shape

    sort_key = np.isnan(vol_array).astype(int)
    order = np.argsort(sort_key, axis=1, kind='stable')
    compressed = np.take_along_axis(vol_array, order, axis=1)

    contract_names_arr = np.array(contract_cols)

    out = pd.DataFrame({'Date': dates})

    if num_contracts > 0:
        out['FirstVolume'] = compressed[:, 0]
        names = contract_names_arr[order[:, 0]]
        out['FirstContract'] = np.where(np.isnan(compressed[:, 0]), '', names)
    else:
        out['FirstVolume'] = np.nan
        out['FirstContract'] = ''

    if num_contracts > 1:
        out['SecondVolume'] = compressed[:, 1]
        names = contract_names_arr[order[:, 1]]
        out['SecondContract'] = np.where(np.isnan(compressed[:, 1]), '', names)
    else:
        out['SecondVolume'] = np.nan
        out['SecondContract'] = ''

    return out

In [ ]:
base_map = build_base_map(individual_symbols)

ccb_symbols = back_adjusted_df['Symbol'].unique().tolist()
cont_to_process = []
needed_contracts = set()
for cont_sym in ccb_symbols:
    base = get_base_from_continuous(cont_sym, base_map)
    if base in base_map:
        cont_to_process.append((cont_sym, base))
        for sym, _ in base_map[base]:
            needed_contracts.add(sym)
    else:
        logging.warning(f"No individual contracts for {cont_sym} (base: {base})")

print(f"Processing {len(cont_to_process)} continuous symbols using {len(needed_contracts)} individual contracts.")

# Extract volume from already-downloaded individual contracts (no re-download needed)
contract_volumes = extract_contract_volumes(individual_df, needed_contracts)
logging.info(f"Extracted volumes for {len(contract_volumes)} contracts from individual_df.")

# Compute first & second contract volumes for each continuous symbol
volume_frames = []
for cont_sym, base in cont_to_process:
    sym_data = back_adjusted_df[back_adjusted_df['Symbol'] == cont_sym]
    if sym_data.empty:
        continue
    vol_df = compute_first_second_volume(sym_data['Date'].values, base_map[base], contract_volumes)
    if vol_df is not None:
        vol_df['Symbol'] = cont_sym
        volume_frames.append(vol_df)

if volume_frames:
    all_volumes = pd.concat(volume_frames, ignore_index=True)
    back_adjusted_with_vol = back_adjusted_df.merge(all_volumes, on=['Date', 'Symbol'], how='left')
    save_to_parquet(back_adjusted_with_vol, PATH_CONTINUOUS_WITH_VOLUMES)
    print(f"Shape: {back_adjusted_with_vol.shape}")
else:
    logging.error("No volume decomposition data computed.")

back_adjusted_with_vol.head(10)

Step 7: Summary and Output Verification

In [ ]:
print(f"All files saved to '{OUTPUT_DIR}/':\n")

outputs = [
    ("01_contract_specs",                      metadata_df,            PATH_CONTRACT_SPECS),
    ("02_individual_contracts_ohlcv",           individual_df,          PATH_INDIVIDUAL_OHLCV),
    ("03_continuous_adjusted_and_unadjusted",   back_adjusted_df,       PATH_CONTINUOUS_ADJ_UNADJ),
    ("04_continuous_with_volume_decomposition", back_adjusted_with_vol, PATH_CONTINUOUS_WITH_VOLUMES),
]

for label, df, path in outputs:
    print(f"  {label}")
    print(f"    Rows: {len(df):,}  |  Columns: {list(df.columns)}")
    print(f"    Path: {path}\n")